# Chunk `voice-iso.mp3` into 10-second files

This notebook uses local `ffmpeg`/`ffprobe` to load metadata from the source MP3 and split it into 10-second chunks.

In [70]:
import json
import shutil
import subprocess
from pathlib import Path

from IPython.display import Audio, display

SOURCE_AUDIO = Path(
    "/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso.mp3"
)
OUTPUT_DIR = SOURCE_AUDIO.with_name("voice-iso_chunks_10s")
CHUNK_SECONDS = 20

if not SOURCE_AUDIO.exists():
    raise FileNotFoundError(f"Missing source audio: {SOURCE_AUDIO}")

if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg is not installed or is not available on PATH")

if shutil.which("ffprobe") is None:
    raise RuntimeError("ffprobe is not installed or is not available on PATH")

SOURCE_AUDIO, OUTPUT_DIR

(PosixPath('/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso.mp3'),
 PosixPath('/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s'))

In [71]:
def run_command(args):
    result = subprocess.run(args, check=True, capture_output=True, text=True)
    return result.stdout


probe_json = run_command(
    [
        "ffprobe",
        "-v",
        "error",
        "-show_format",
        "-show_streams",
        "-of",
        "json",
        str(SOURCE_AUDIO),
    ]
)

metadata = json.loads(probe_json)
duration = float(metadata["format"]["duration"])
audio_stream = next(
    stream for stream in metadata["streams"] if stream["codec_type"] == "audio"
)

print(f"Source: {SOURCE_AUDIO}")
print(f"Duration: {duration:.2f}s")
print(f"Codec: {audio_stream.get('codec_name', 'unknown')}")
print(f"Sample rate: {audio_stream.get('sample_rate', 'unknown')} Hz")
print(f"Channels: {audio_stream.get('channels', 'unknown')}")

Source: /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso.mp3
Duration: 180.74s
Codec: mp3
Sample rate: 44100 Hz
Channels: 2


In [72]:
display(Audio(filename=str(SOURCE_AUDIO)))

In [73]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for old_chunk in OUTPUT_DIR.glob("voice-iso_chunk_*.mp3"):
    old_chunk.unlink()

output_pattern = OUTPUT_DIR / "voice-iso_chunk_%03d.mp3"

cmd = [
    "ffmpeg",
    "-hide_banner",
    "-y",
    "-i",
    str(SOURCE_AUDIO),
    "-map",
    "0:a:0",
    "-f",
    "segment",
    "-segment_time",
    str(CHUNK_SECONDS),
    "-reset_timestamps",
    "1",
    "-c:a",
    "libmp3lame",
    "-q:a",
    "2",
    str(output_pattern),
]

subprocess.run(cmd, check=True)

chunks = sorted(OUTPUT_DIR.glob("voice-iso_chunk_*.mp3"))
print(f"Created {len(chunks)} chunks in {OUTPUT_DIR}")
for chunk in chunks:
    print(chunk.name)

Input #0, mp3, from '/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso.mp3':
  Metadata:
    encoder         : Lavf58.76.100
  Duration: 00:03:00.74, start: 0.025057, bitrate: 192 kb/s
  Stream #0:0: Audio: mp3 (mp3float), 44100 Hz, stereo, fltp, 192 kb/s
      Metadata:
        encoder         : Lavc58.13
Stream mapping:
  Stream #0:0 -> #0:0 (mp3 (mp3float) -> mp3 (libmp3lame))
Press [q] to stop, [?] for help
[segment @ 0x749410280] Opening '/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s/voice-iso_chunk_000.mp3' for writing
Output #0, segment, to '/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s/voice-iso_chunk_%03d.mp3':
  Metadata:
    encoder         : Lavf61.7.100
  Stream #0:0: Audio: mp3, 44100 Hz, stereo, fltp
      Metadata:
        encoder         : Lavc61.19.101 libmp3lame
[segment @ 0x749410280] Opening '/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s/voice-iso_chunk_001.mp3' for

Created 10 chunks in /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s
voice-iso_chunk_000.mp3
voice-iso_chunk_001.mp3
voice-iso_chunk_002.mp3
voice-iso_chunk_003.mp3
voice-iso_chunk_004.mp3
voice-iso_chunk_005.mp3
voice-iso_chunk_006.mp3
voice-iso_chunk_007.mp3
voice-iso_chunk_008.mp3
voice-iso_chunk_009.mp3


[segment @ 0x749410280] Opening '/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s/voice-iso_chunk_008.mp3' for writing
[segment @ 0x749410280] Opening '/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s/voice-iso_chunk_009.mp3' for writing
[libmp3lame @ 0x748874a80] Trying to remove 1152 samples, but the queue is empty
[out#0/segment @ 0x749414240] video:0KiB audio:3229KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: unknown
size=N/A time=00:03:00.68 bitrate=N/A speed= 134x    


In [29]:
def get_duration_seconds(audio_path):
    output = run_command(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "format=duration",
            "-of",
            "default=noprint_wrappers=1:nokey=1",
            str(audio_path),
        ]
    )
    return float(output.strip())


chunk_report = [
    {"file": chunk.name, "duration_seconds": round(get_duration_seconds(chunk), 3)}
    for chunk in chunks
]

chunk_report

[{'file': 'voice-iso_chunk_000.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_001.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_002.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_003.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_004.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_005.mp3', 'duration_seconds': 9.979},
 {'file': 'voice-iso_chunk_006.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_007.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_008.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_009.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_010.mp3', 'duration_seconds': 9.979},
 {'file': 'voice-iso_chunk_011.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_012.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_013.mp3', 'duration_seconds': 10.005},
 {'file': 'voice-iso_chunk_014.mp3', 'duration_seconds': 10.005},
 {'file': 'v

In [30]:
for chunk in chunks[:3]:
    print(chunk.name)
    display(Audio(filename=str(chunk)))

voice-iso_chunk_000.mp3


voice-iso_chunk_001.mp3


voice-iso_chunk_002.mp3


## Test VieNeu-TTS with reference audio

This section loads the first three 10-second audio chunks and their matching transcripts from `resource/sub`, then uses each pair as a reference voice cloning prompt.

In [31]:
REFERENCE_AUDIO_DIR = Path(
    "/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s"
)
REFERENCE_TEXT_DIR = Path("/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/sub")
VIENEU_OUTPUT_DIR = Path(
    "/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests"
)

reference_audio_files = sorted(REFERENCE_AUDIO_DIR.glob("voice-iso_chunk_*.mp3"))[:3]
reference_text_files = [REFERENCE_TEXT_DIR / f"{idx}.txt" for idx in range(1, 4)]

missing_audio = [path for path in reference_audio_files if not path.exists()]
missing_text = [path for path in reference_text_files if not path.exists()]

if len(reference_audio_files) < 3:
    raise FileNotFoundError(
        f"Expected at least 3 reference audio chunks in {REFERENCE_AUDIO_DIR}"
    )
if missing_audio or missing_text:
    raise FileNotFoundError(
        {"missing_audio": missing_audio, "missing_text": missing_text}
    )

references = []
for audio_path, text_path in zip(reference_audio_files, reference_text_files):
    references.append(
        {
            "audio": audio_path,
            "text_file": text_path,
            "text": text_path.read_text(encoding="utf-8").strip(),
            "duration_seconds": round(get_duration_seconds(audio_path), 3),
        }
    )

for idx, ref in enumerate(references, start=1):
    print(f"Reference {idx}")
    print(f"  audio: {ref['audio']}")
    print(f"  text:  {ref['text']}")
    print(f"  duration: {ref['duration_seconds']}s")
    display(Audio(filename=str(ref["audio"])))

Reference 1
  audio: /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s/voice-iso_chunk_000.mp3
  text:  Hồi mình còn học đại học, có những ngày trong túi chẳng có tiền tiêu, thứ duy nhất mà mình còn là xăng trong con xe cắp cũ. Và cứ thế một buổi chiều buồn
  duration: 10.005s


Reference 2
  audio: /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s/voice-iso_chunk_001.mp3
  text:  buồn bâng khuâng mình lôi xe ra đi lang thang chẳng biết đi đâu cứ vậy mà đi thôi. Hồi đó xăng còn đủ sức chi trả chứ nếu là bây giờ thì thú thực mình
  duration: 10.005s


Reference 3
  audio: /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/voice-iso_chunks_10s/voice-iso_chunk_002.mp3
  text:  cũng không dám chắc là mình dám làm điều đó hay không nữa. Nghe có vẻ buồn cười nhưng mình thực sự đã rất tận hưởng chuyến đi lang thang đó. Đấy là lần duy nhất mà mình đi tay không
  duration: 10.005s


In [32]:
import importlib.util

if importlib.util.find_spec("vieneu") is None:
    raise ModuleNotFoundError(
        "Install VieNeu-TTS first, then rerun this cell. Example: pip install vieneu",
    )
if importlib.util.find_spec("neucodec") is None:
    raise ModuleNotFoundError(
        "Reference audio cloning needs the PyTorch NeuCodec encoder. Run: pip install neucodec",
    )

from vieneu import Vieneu

tts = Vieneu(
    codec_repo="neuphonic/neucodec",
    codec_device="cpu",
)

if not hasattr(tts.codec, "encode_code"):
    raise RuntimeError(
        f"Loaded codec {type(tts.codec).__name__} cannot encode reference audio. "
        'Use codec_repo="neuphonic/neucodec" or codec_repo="neuphonic/distill-neucodec".',
    )

/Users/rzy/Desktop/ProjectWithTien/cv-helper/cv-fit-app/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not suppo

In [ ]:
VIENEU_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

target_text = (
    "Chào bạn đến với buổi phỏng vấn hôm nay. Trước hết hãy giới thiệu về bản thân"
)


def infer_vieneu_with_reference(tts_client, text, ref_audio, ref_text):
    ref_codes = tts_client.encode_reference(str(ref_audio))
    return tts_client.infer(
        text=text,
        ref_codes=ref_codes,
        ref_text=ref_text,
    )


generated_files = []
for idx, ref in enumerate(references, start=1):
    output_path = VIENEU_OUTPUT_DIR / f"vieneu_ref_{idx:02d}.wav"
    audio = infer_vieneu_with_reference(
        tts_client=tts,
        text=target_text,
        ref_audio=ref["audio"],
        ref_text=ref["text"],
    )
    tts.save(audio, str(output_path))
    generated_files.append(output_path)
    print(f"Saved {output_path}")
    display(Audio(filename=str(output_path)))

generated_files

Saved /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/vieneu_ref_01.wav


Saved /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/vieneu_ref_02.wav


Saved /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/vieneu_ref_03.wav


[PosixPath('/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/vieneu_ref_01.wav'),
 PosixPath('/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/vieneu_ref_02.wav'),
 PosixPath('/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/vieneu_ref_03.wav')]

## Test VieNeu-TTS with all reference samples

VieNeu standard mode accepts one reference voice prompt. To use multiple samples, concatenate the audio files and combine their transcripts into one reference pair. Keep generated text short because long references consume model context.

In [34]:
COMBINED_REF_AUDIO = VIENEU_OUTPUT_DIR / "combined_reference_3_samples.wav"
COMBINED_REF_LIST = VIENEU_OUTPUT_DIR / "combined_reference_inputs.txt"

VIENEU_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_REF_LIST.write_text(
    "".join(f"file '{ref['audio']}'\n" for ref in references),
    encoding="utf-8",
)

subprocess.run(
    [
        "ffmpeg",
        "-hide_banner",
        "-y",
        "-f",
        "concat",
        "-safe",
        "0",
        "-i",
        str(COMBINED_REF_LIST),
        "-ac",
        "1",
        "-ar",
        "16000",
        str(COMBINED_REF_AUDIO),
    ],
    check=True,
)

combined_ref_text = " ".join(ref["text"] for ref in references)

print(f"Combined reference audio: {COMBINED_REF_AUDIO}")
print(f"Duration: {get_duration_seconds(COMBINED_REF_AUDIO):.3f}s")
print(f"Text: {combined_ref_text}")
display(Audio(filename=str(COMBINED_REF_AUDIO)))

Combined reference audio: /Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/combined_reference_3_samples.wav
Duration: 29.966s
Text: Hồi mình còn học đại học, có những ngày trong túi chẳng có tiền tiêu, thứ duy nhất mà mình còn là xăng trong con xe cắp cũ. Và cứ thế một buổi chiều buồn buồn bâng khuâng mình lôi xe ra đi lang thang chẳng biết đi đâu cứ vậy mà đi thôi. Hồi đó xăng còn đủ sức chi trả chứ nếu là bây giờ thì thú thực mình cũng không dám chắc là mình dám làm điều đó hay không nữa. Nghe có vẻ buồn cười nhưng mình thực sự đã rất tận hưởng chuyến đi lang thang đó. Đấy là lần duy nhất mà mình đi tay không


Input #0, concat, from '/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/combined_reference_inputs.txt':
  Duration: N/A, start: -0.025057, bitrate: 143 kb/s
  Stream #0:0: Audio: mp3 (mp3float), 44100 Hz, stereo, fltp, 143 kb/s
      Metadata:
        encoder         : Lavc61.19
Stream mapping:
  Stream #0:0 -> #0:0 (mp3 (mp3float) -> pcm_s16le (native))
Press [q] to stop, [?] for help
Output #0, wav, to '/Users/rzy/Desktop/ProjectWithTien/cv-helper/resource/vieneu_reference_tests/combined_reference_3_samples.wav':
  Metadata:
    ISFT            : Lavf61.7.100
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 16000 Hz, mono, s16, 256 kb/s
      Metadata:
        encoder         : Lavc61.19.101 pcm_s16le
[out#0/wav @ 0xb210240c0] video:0KiB audio:936KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.008134%
size=     937KiB time=00:00:30.02 bitrate= 255.5kbits/s speed= 919x    


In [35]:
combined_output_path = VIENEU_OUTPUT_DIR / "vieneu_ref_all_3_samples.wav"
combined_target_text = (
    "Mình đang kiểm tra giọng nói khi dùng cả ba đoạn tham chiếu cùng lúc."
)

combined_ref_codes = tts.encode_reference(str(COMBINED_REF_AUDIO))
combined_audio = tts.infer(
    text=combined_target_text,
    ref_codes=combined_ref_codes,
    ref_text=combined_ref_text,
    max_chars=96,
)

tts.save(combined_audio, str(combined_output_path))
print(f"Saved {combined_output_path}")
display(Audio(filename=str(combined_output_path)))

combined_output_path

ValueError: Requested tokens (2153) exceed context window of 2048

In [1]:
import requests

In [2]:
# Test voices endpoint
response = requests.get("https://divorcee-work-pessimism.ngrok-free.dev/api/tts/voices")
print("Voices:", response.json())

Voices: {'success': True, 'count': 7, 'voices': [{'description': 'Thanh Bình (nam miền Bắc)', 'voice_id': 'Binh'}, {'description': 'Phạm Tuyên (nam miền Bắc)', 'voice_id': 'Tuyen'}, {'description': 'Xuân Vĩnh (nam miền Nam)', 'voice_id': 'Vinh'}, {'description': 'Thục Đoan (nữ miền Nam)', 'voice_id': 'Doan'}, {'description': 'Trúc Ly (nữ miền Bắc)', 'voice_id': 'Ly'}, {'description': 'Thái Sơn (nam miền Nam)', 'voice_id': 'Sơn'}, {'description': 'Bích Ngọc (nữ miền Bắc)', 'voice_id': 'Ngoc'}]}


In [29]:
url = "https://divorcee-work-pessimism.ngrok-free.dev/api/tts/generate"

data = {
    "text": "Chủ đích của sự không chủ đích. Người quan sát thành phố như đọc một cuốn sách, mà không ai đã viết sẵn cho họ.",
}
response = requests.post(url, json=data)

In [30]:
# Check if request was successful
response.raise_for_status()  # throws HTTPError on 4xx/5xx

# Save to local file
output_path = "downloaded_audio_1.mp3"
with open(output_path, "wb") as f:
    f.write(
        response.content
    )  # or response.iter_content(chunk_size=8192) for large files

print(f"✅ Audio saved to: {output_path}")

✅ Audio saved to: downloaded_audio_1.mp3


In [21]:
rep_json = response.json()
rep_json

{'success': True,
 'message': 'Speech synthesized successfully',
 'audio_file': 'outputs/tts/tts_6546.wav',
 'voice_id': 'default'}

In [22]:
filename = rep_json["audio_file"].split("/")[-1]

In [23]:
url = "https://divorcee-work-pessimism.ngrok-free.dev/api/tts/audio/" + filename
response = requests.get(url)

In [24]:
# Check if request was successful
response.raise_for_status()  # throws HTTPError on 4xx/5xx

# Save to local file
output_path = "downloaded_audio.mp3"
with open(output_path, "wb") as f:
    f.write(
        response.content
    )  # or response.iter_content(chunk_size=8192) for large files

print(f"✅ Audio saved to: {output_path}")

✅ Audio saved to: downloaded_audio.mp3


In [34]:
url = "https://divorcee-work-pessimism.ngrok-free.dev/api/tts/generate"
response = requests.post(
    url,
    json={
        "text": "Anh/chị có thể giới thiệu ngắn gọn về bản thân, tập trung vào hành trình từ khi học đại học đến hiện tại, và lý do nào khiến anh/chị chọn theo đuổi lĩnh vực Machine Learning / AI?",
        "self_clone": True,
    },
)

response.raise_for_status()  # throws HTTPError on 4xx/5xx
# Save to local file
output_path = "downloaded_audio.mp3"
with open(output_path, "wb") as f:
    f.write(
        response.content
    )  # or response.iter_content(chunk_size=8192) for large files

print(f"✅ Audio saved to: {output_path}")

✅ Audio saved to: downloaded_audio.mp3
